# STM32F429 CRC (Cyclic Redundancy Check) Peripheral Tutorial

This comprehensive tutorial covers the CRC (Cyclic Redundancy Check) peripheral on the STM32F429 microcontroller. The CRC peripheral provides hardware-accelerated data integrity verification using various CRC standards.

## Table of Contents
1. [CRC Overview](#crc-overview)
2. [Hardware Architecture](#hardware-architecture)
3. [Register Configuration](#register-configuration)
4. [API Reference](#api-reference)
5. [Basic Usage Examples](#basic-usage-examples)
6. [Advanced Features](#advanced-features)
7. [Performance Considerations](#performance-considerations)
8. [Troubleshooting](#troubleshooting)

---

## CRC Overview

The CRC (Cyclic Redundancy Check) peripheral in STM32F429 provides:

### Key Features
- **Hardware CRC Accelerator**: Dedicated hardware for high-performance CRC calculations
- **Multiple CRC Standards**: Support for CRC-32, CRC-32C, CRC-16, CRC-16-CCITT, and CRC-8
- **Flexible Configuration**: Configurable polynomials, initial values, and data formats
- **Data Accumulation**: Process large datasets in chunks
- **Software Fallback**: Software implementation for unsupported polynomials

### Applications
- Data integrity verification
- Communication protocol checksums
- File system integrity
- Firmware validation
- Network packet validation

### Supported Standards
- **CRC-32**: Ethernet, ZIP, PNG (0x04C11DB7)
- **CRC-32C**: iSCSI, SCTP (0x1EDC6F41)
- **CRC-16**: Modbus, USB (0x8005)
- **CRC-16-CCITT**: X.25, V.41 (0x1021)
- **CRC-8**: Dallas 1-Wire (0x07)

## Hardware Architecture

### STM32F429 CRC Block Diagram

```
Data Input (32-bit)
       |
       v
    +----------+
    |   CRC    |
    |  Engine  |
    +----------+
       |
       v
   CRC Result (32-bit)
```

### Key Characteristics
- **Fixed Polynomial**: Hardware uses CRC-32 (0x04C11DB7)
- **32-bit Data Processing**: Optimized for 32-bit word operations
- **Hardware Acceleration**: ~15x faster than software implementation
- **Low Power**: Minimal power consumption during operation

### Memory Mapping
- **Base Address**: 0x40023000
- **Size**: 0x400 bytes
- **Bus**: AHB1

## Register Configuration

### CRC Registers Overview

The STM32F429 CRC peripheral uses the following registers:

#### CRC_DR (Data Register) - 0x40023000
- **Purpose**: Holds CRC calculation result and accepts input data
- **Access**: Read/Write
- **Reset Value**: 0xFFFFFFFF
- **Bit Fields**:
  - Bits[31:0]: CRC result or input data

#### CRC_IDR (Independent Data Register) - 0x40023004
- **Purpose**: General-purpose 8-bit data register
- **Access**: Read/Write
- **Reset Value**: 0x00
- **Bit Fields**:
  - Bits[7:0]: Independent data

#### CRC_CR (Control Register) - 0x40023008
- **Purpose**: Control CRC peripheral operation
- **Access**: Read/Write
- **Reset Value**: 0x00
- **Bit Fields**:
  - Bit 0: RESET - Reset CRC calculation

### Register Setup Sequence

```c
// Enable CRC peripheral clock
RCC->AHB1ENR |= RCC_AHB1ENR_CRCEN;

// Reset CRC calculation (optional - resets to 0xFFFFFFFF)
CRC->CR |= CRC_CR_RESET;

// Write data to CRC_DR register
CRC->DR = data_word;

// Read CRC result
uint32_t crc_result = CRC->DR;
```

### Hardware Limitations
- **Fixed Polynomial**: Cannot change CRC polynomial in hardware
- **No Bit Reversal**: Hardware doesn't support input/output bit reversal
- **32-bit Only**: Hardware processes data in 32-bit chunks only

## API Reference

### Configuration Structures

```c
typedef struct {
    CRC_Method method;           // Hardware or software calculation
    uint32_t polynomial;         // CRC polynomial
    uint32_t init_value;         // Initial CRC value
    bool input_reverse;          // Reverse input data bytes
    bool output_reverse;         // Reverse output CRC
    CRC_DataFormat input_format; // Input data format
    CRC_DataFormat output_format;// Output data format
    bool xor_output;             // XOR output with value
    uint32_t xor_value;          // Value to XOR with output
} CRC_Config;
```

### Core Functions

#### Initialization
```c
HAL_StatusTypeDef CRC_Init(const CRC_Config *config);
HAL_StatusTypeDef CRC_DeInit(void);
void CRC_GetDefaultConfig(CRC_Config *config);
```

#### Calculation Functions
```c
HAL_StatusTypeDef CRC_Calculate(const uint8_t *data, uint32_t size, uint32_t *crc);
HAL_StatusTypeDef CRC_Calculate32(const uint32_t *data, uint32_t size, uint32_t *crc);
HAL_StatusTypeDef CRC_Accumulate(const uint8_t *data, uint32_t size, uint32_t *crc);
HAL_StatusTypeDef CRC_Reset(void);
```

#### Configuration Functions
```c
HAL_StatusTypeDef CRC_SetPolynomial(uint32_t polynomial);
HAL_StatusTypeDef CRC_SetInitValue(uint32_t init_value);
HAL_StatusTypeDef CRC_GetStatus(CRC_Status *status);
```

#### Callback Functions
```c
void CRC_RegisterCompleteCallback(void (*callback)(uint32_t crc_value));
void CRC_RegisterErrorCallback(void (*callback)(CRC_ErrorType error));
```

## Basic Usage Examples

### Example 1: Basic CRC-32 Calculation

```c
#include "Peripherals/CRC/crc.h"

int main(void) {
    // Initialize system
    HAL_Init();
    
    // Get default configuration
    CRC_Config config;
    CRC_GetDefaultConfig(&config);
    
    // Initialize CRC
    if (CRC_Init(&config) != HAL_OK) {
        // Handle error
        while(1);
    }
    
    // Calculate CRC for data
    uint8_t data[] = {0x11, 0x22, 0x33, 0x44, 0x55};
    uint32_t crc_value;
    
    if (CRC_Calculate(data, sizeof(data), &crc_value) == HAL_OK) {
        // CRC calculated successfully
        // crc_value contains the CRC-32 result
    }
    
    while(1);
}
```

### Example 2: 32-bit Word Calculation (Hardware Optimized)

```c
// Example with 32-bit aligned data
uint32_t data32[] = {0x11223344, 0x55667788, 0x99AABBCC};
uint32_t crc_result;

if (CRC_Calculate32(data32, 3, &crc_result) == HAL_OK) {
    // Hardware-accelerated calculation
    // crc_result = calculated CRC value
}
```

### Example 3: Data Accumulation

```c
// Process large data in chunks
uint32_t accumulated_crc = 0;
uint8_t chunk1[] = {0x11, 0x22, 0x33, 0x44};
uint8_t chunk2[] = {0x55, 0x66, 0x77, 0x88};

CRC_Accumulate(chunk1, sizeof(chunk1), &accumulated_crc);
CRC_Accumulate(chunk2, sizeof(chunk2), &accumulated_crc);

// accumulated_crc now contains CRC of combined data
```

## Advanced Features

### Custom CRC Configuration

```c
// Configure for CRC-16-CCITT
CRC_Config custom_config;
custom_config.method = CRC_METHOD_SOFTWARE;  // Hardware doesn't support custom polynomials
custom_config.polynomial = CRC_POLY_CRC16_CCITT;  // 0x1021
custom_config.init_value = 0xFFFF;
custom_config.input_reverse = false;
custom_config.output_reverse = false;
custom_config.input_format = CRC_FORMAT_8BIT;
custom_config.output_format = CRC_FORMAT_16BIT;
custom_config.xor_output = true;
custom_config.xor_value = 0x0000;

CRC_Init(&custom_config);
```

### Callback Usage

```c
// Completion callback
void crc_complete_callback(uint32_t crc_value) {
    printf("CRC calculation complete: 0x%08X\n", crc_value);
}

// Error callback
void crc_error_callback(CRC_ErrorType error) {
    printf("CRC error occurred: %d\n", error);
}

// Register callbacks
CRC_RegisterCompleteCallback(crc_complete_callback);
CRC_RegisterErrorCallback(crc_error_callback);
```

### Status Monitoring

```c
CRC_Status status;
CRC_GetStatus(&status);

printf("CRC Status:\n");
printf("  Initialized: %s\n", status.initialized ? "Yes" : "No");
printf("  Method: %s\n", status.current_method == CRC_METHOD_HARDWARE ? "Hardware" : "Software");
printf("  Last CRC: 0x%08X\n", status.last_calculated_crc);
printf("  Calculations: %lu\n", status.calculation_count);
printf("  Hardware usage: %lu\n", status.hardware_usage_count);
printf("  Software usage: %lu\n", status.software_usage_count);
```

## Performance Considerations

### Hardware vs Software Performance

| Method | Data Size | Time | Relative Speed |
|--------|-----------|------|----------------|
| Hardware CRC | 1KB | ~2.5μs | 15x faster |
| Software CRC | 1KB | ~37μs | 1x baseline |

### Optimization Tips

1. **Use Hardware When Possible**
   ```c
   // Hardware is fastest for CRC-32
   config.method = CRC_METHOD_HARDWARE;
   config.polynomial = CRC_POLY_CRC32;
   ```

2. **32-bit Aligned Data**
   ```c
   // Use CRC_Calculate32 for word-aligned data
   uint32_t aligned_data[] = {0x12345678, 0x9ABCDEF0};
   CRC_Calculate32(aligned_data, 2, &crc);
   ```

3. **Data Accumulation for Large Datasets**
   ```c
   // Process in chunks to avoid memory constraints
   #define CHUNK_SIZE 1024
   uint32_t crc = 0;
   for (size_t i = 0; i < total_size; i += CHUNK_SIZE) {
       size_t chunk = (total_size - i) > CHUNK_SIZE ? CHUNK_SIZE : (total_size - i);
       CRC_Accumulate(&data[i], chunk, &crc);
   }
   ```

### Memory Usage
- **Hardware Method**: Minimal RAM usage (~100 bytes)
- **Software Method**: Same RAM usage, higher CPU utilization
- **Maximum Data Size**: 4096 bytes per calculation
- **Stack Usage**: ~256 bytes for local variables

## Troubleshooting

### Common Issues and Solutions

#### 1. CRC Calculation Returns Wrong Values

**Problem**: CRC results don't match expected values

**Solutions**:
- Verify polynomial configuration
- Check initial value settings
- Ensure correct data endianness
- Confirm XOR output settings

**Debug Code**:
```c
// Test with known data
uint8_t test_data[] = {0x31, 0x32, 0x33, 0x34};  // "1234"
uint32_t crc;
CRC_Calculate(test_data, 4, &crc);
// Expected CRC-32: 0x9BE3E0A3
printf("CRC: 0x%08X\n", crc);
```

#### 2. Hardware CRC Not Working

**Problem**: Hardware CRC returns HAL_ERROR

**Solutions**:
- Enable CRC peripheral clock: `RCC->AHB1ENR |= RCC_AHB1ENR_CRCEN;`
- Check RCC configuration
- Verify HAL_CRC_Init() return value
- Use software fallback if hardware fails

#### 3. Data Size Limitations

**Problem**: Large data arrays cause errors

**Solutions**:
- Use data accumulation for large datasets
- Process data in chunks ≤ 4096 bytes
- Implement streaming CRC calculation

**Chunked Processing**:
```c
#define CRC_CHUNK_SIZE 4096
uint32_t calculate_large_crc(const uint8_t *data, size_t size) {
    uint32_t crc = 0;
    for (size_t i = 0; i < size; i += CRC_CHUNK_SIZE) {
        size_t chunk_size = (size - i) > CRC_CHUNK_SIZE ? CRC_CHUNK_SIZE : (size - i);
        CRC_Accumulate(&data[i], chunk_size, &crc);
    }
    return crc;
}
```

#### 4. Callback Functions Not Called

**Problem**: Registered callbacks aren't triggered

**Solutions**:
- Ensure callbacks are registered before calculations
- Check callback function signatures
- Verify CRC initialization completed successfully
- Use polling instead of callbacks for simple applications

### Error Codes

| Error Code | Description | Solution |
|------------|-------------|----------|
| CRC_ERROR_NONE | No error | N/A |
| CRC_ERROR_INVALID_PARAM | Invalid parameters | Check function arguments |
| CRC_ERROR_DATA_SIZE | Data too large | Use data accumulation |
| CRC_ERROR_HARDWARE | Hardware peripheral error | Check RCC and peripheral state |
| CRC_ERROR_TIMEOUT | Operation timeout | Check system clock configuration |

### Debug Information

```c
// Enable detailed logging
void crc_debug_info(void) {
    CRC_Status status;
    CRC_GetStatus(&status);
    
    printf("=== CRC Debug Info ===\n");
    printf("Initialized: %s\n", status.initialized ? "Yes" : "No");
    printf("Method: %s\n", status.current_method == CRC_METHOD_HARDWARE ? "Hardware" : "Software");
    printf("Last Error: %d\n", status.last_error);
    printf("Total Calculations: %lu\n", status.calculation_count);
    printf("Hardware Usage: %lu\n", status.hardware_usage_count);
    printf("Software Usage: %lu\n", status.software_usage_count);
}
```

### Testing CRC Implementation

```c
// Test vectors for CRC-32
typedef struct {
    const char *data;
    uint32_t expected_crc;
} crc_test_vector_t;

crc_test_vector_t test_vectors[] = {
    {"123456789", 0xCBF43926},
    {"The quick brown fox jumps over the lazy dog", 0x414FA339},
    {"", 0x00000000},  // Empty string
};

void test_crc_implementation(void) {
    CRC_Config config;
    CRC_GetDefaultConfig(&config);
    CRC_Init(&config);
    
    for (int i = 0; i < sizeof(test_vectors)/sizeof(test_vectors[0]); i++) {
        uint32_t crc;
        CRC_Calculate((uint8_t*)test_vectors[i].data, 
                      strlen(test_vectors[i].data), &crc);
        
        if (crc == test_vectors[i].expected_crc) {
            printf("Test %d PASSED\n", i);
        } else {
            printf("Test %d FAILED: expected 0x%08X, got 0x%08X\n", 
                   i, test_vectors[i].expected_crc, crc);
        }
    }
}
```

## Summary

The STM32F429 CRC peripheral provides efficient data integrity verification with both hardware acceleration and software flexibility. Key takeaways:

### Hardware Features
- Fixed CRC-32 polynomial (0x04C11DB7)
- 32-bit data processing
- ~15x performance improvement over software
- Low power consumption

### Software Features
- Multiple CRC standards support
- Configurable polynomials and parameters
- Bit reversal and XOR operations
- Data accumulation for large datasets

### Best Practices
1. Use hardware CRC for CRC-32 calculations
2. Use 32-bit aligned data when possible
3. Implement data accumulation for large files
4. Register error callbacks for robust applications
5. Test with known CRC values for validation

### Performance Optimization
- Hardware: ~2.5μs per KB
- Software: ~37μs per KB
- Memory efficient (< 1KB RAM)
- Suitable for real-time applications

This tutorial provides comprehensive coverage of CRC implementation on STM32F429, from basic usage to advanced features and troubleshooting techniques.